# Hotspot Gene Module Discovery in Malignant Cells

Subsets the epithelial/tumour compartment to malignant clusters, runs
Hotspot to discover co-expressed gene
modules along the FDG trajectory, scores each cell for module activity.

In [ ]:
# !pip install --quiet scanpy leidenalg igraph louvain scikit-misc mplscience hotspotsc decoupler

In [ ]:
import os

import matplotlib.pyplot as plt
import mplscience
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from hotspot import Hotspot
from scipy.stats import pearsonr, spearmanr

import decoupler as dc

## Config

In [ ]:
input_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06.h5ad"
output_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_hvg2000.h5ad"

hotspot_output_dir = "./hotspot_subset_cluster"
enrichment_output_dir = "./enrichment_results/progeny"

malignant_label = "Malignant"
n_top_hvgs = 2000

# Hotspot parameters
hotspot_layer = "raw_dense"
hotspot_model = "danb"
hotspot_latent_key = "X_draw_graph_fa"  # FDG layout computed upstream
hotspot_n_neighbors = 30
hotspot_fdr_threshold = 0.05
hotspot_min_gene_threshold = 30
hotspot_jobs = 96

# display order for module-by-donor/response heatmaps
module_order = [
    "Module 1", "Module 3", "Module 2", "Module 8", "Module 9", "Module 14",
    "Module 5", "Module 6", "Module 4", "Module 12", "Module 7", "Module 13",
    "Module 10", "Module 11", "Module 15",
]
donor_order = [
    "TN004_AN", "TN006_AN", "TN007_AN", "TN002_AN", "TN001_AN", "TN005_AN", "TN003_AN", "TN009_AN",
    "TN008", "TN004", "TN006", "TN007", "TN002", "TN001", "TN005", "TN003", "TN009",
    "P003", "P002", "P008", "P006", "P012", "P005", "P013", "P001", "P009", "P011",
    "R006", "R003", "R001", "R010", "R009", "R007", "R005",
]

progeny_pathways = [
    "Androgen", "EGFR", "Estrogen", "Hypoxia", "JAK-STAT", "MAPK",
    "NFkB", "PI3K", "TGFb", "TNFa", "Trail", "VEGF", "WNT", "p53",
]
correlation_method = "spearman"  # "spearman" or "pearson"
n_response_markers = 15  # top TF markers per RECIST group for the final dotplot

os.makedirs(hotspot_output_dir, exist_ok=True)
os.makedirs(enrichment_output_dir, exist_ok=True)

## Load data and subset to malignant clusters

In [ ]:
adata = sc.read(input_path)
adata = adata[adata.obs["cell_type_new"] == malignant_label].copy()
adata.obs["ID_2"].value_counts()

## Preprocess and select highly variable genes

In [ ]:
adata.layers["raw_counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# subset=True restricts to HVGs for Hotspot; use subset=False to keep all genes
sc.pp.highly_variable_genes(adata, n_top_genes=n_top_hvgs, flavor="seurat_v3", subset=True)
adata

## Run Hotspot to discover co-expressed gene modules

In [ ]:
adata.layers[hotspot_layer] = adata.layers["raw_counts"]
if hasattr(adata.layers[hotspot_layer], "toarray"):
    adata.layers[hotspot_layer] = adata.layers[hotspot_layer].toarray()

hs = Hotspot(adata, layer_key=hotspot_layer, model=hotspot_model, latent_obsm_key=hotspot_latent_key)
hs.create_knn_graph(weighted_graph=False, n_neighbors=hotspot_n_neighbors)
hs_results = hs.compute_autocorrelations()

hs_genes = hs_results.loc[hs_results.FDR < hotspot_fdr_threshold].index
local_correlations = hs.compute_local_correlations(hs_genes, jobs=hotspot_jobs)

modules = hs.create_modules(
    min_gene_threshold=hotspot_min_gene_threshold, core_only=False, fdr_threshold=hotspot_fdr_threshold
)
print(modules.value_counts())

In [ ]:
module_scores = hs.calculate_module_scores()
results = hs.results.join(hs.modules)
results.to_csv(os.path.join(hotspot_output_dir, "malignant_hotspot_modules.csv"), index=True)

plt.rcParams["figure.figsize"] = (15.0, 12.0)
hs.plot_local_correlations()
plt.savefig(os.path.join(hotspot_output_dir, "hotspot_malignant_local_correlations.svg"), dpi=100, bbox_inches="tight")
plt.show()

## Attach module scores to cells

In [ ]:
module_cols = []
for c in module_scores.columns:
    key = f"Module {c}"
    adata.obs[key] = module_scores[c]
    module_cols.append(key)

## Visualize module activity by cell state and FDG layout

In [ ]:
with mplscience.style_context():
    sc.pl.draw_graph(adata, color=module_cols, frameon=False, vmin=-1, vmax=1)

sc.pl.matrixplot(
    adata, var_names=module_cols, groupby="cell_state",
    dendrogram=True, standard_scale="var", cmap="viridis",
)

In [ ]:
for mod in module_cols:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=adata.obs, x="cell_state", y=mod)
    plt.xticks(rotation=45)
    plt.title(f"{mod} score across cell states")
    plt.tight_layout()
    plt.show()

## Save malignant Hotspot output

In [ ]:
adata.X = adata.layers["raw_counts"].copy()
if "log1p" in adata.uns:
    del adata.uns["log1p"]
if "dendrogram_cell_state" in adata.uns:
    del adata.uns["dendrogram_cell_state"]

adata.write(output_path)
adata

## Module activity by RECIST response and donor

In [ ]:
present_donor_order = [d for d in donor_order if d in adata.obs["ID_2"].unique()]
adata.obs["ID_2"] = adata.obs["ID_2"].astype("category").cat.reorder_categories(present_donor_order, ordered=True)

sc.pl.matrixplot(
    adata, var_names=module_order, groupby="recist_v3",
    standard_scale="var", cmap="GnBu", dendrogram=False, swap_axes=True, figsize=(3, 4),
    save="heatmap_malignant_subset_modules.pdf",
)
sc.pl.matrixplot(
    adata, var_names=module_order, groupby="ID_2",
    standard_scale="var", cmap="GnBu", dendrogram=False, swap_axes=True, figsize=(7, 4),
    save="heatmap_malignant_subset_by_ID_modules.pdf",
)